# Optimisation des performances

---

- **Projet 8 :** Confirmez vos compétences en MLOps (Partie 2/2)
- **Auteur :** Justine Tranchant
- **Date :** avril 2026

---

## Objectif

Mesurer les temps de réponse actuels, identifier les goulots principaux, tester deux optimisations simples et décider ce qui est retenu.
Les chiffres viennent des JSON déjà produits dans `perf/results/`.

## Résultats clés

- **Baseline** : moteur rapide (< 1 ms), temps perçu utilisateur dominé par la couche autour (~122 ms local, ~372 ms préprod).
- **Goulots** : cProfile montre que le modèle n'est pas le principal goulot — overhead sklearn + pandas au-dessus de LightGBM.
- **ONNX Runtime** : optimisation moteur validée. Scores équivalents, inférence ~50× plus rapide, gain absolu ~0.6 ms.
- **PostgreSQL async** : optimisation applicative retenue. Gain net en local (~918 ms gagnées en médiane, intégrité des logs vérifiée), non confirmé sur cette préprod Render.
- **Décision** : deux optimisations conservées, complémentaires (moteur + chemin applicatif), avec lecture prudente de la préprod.

In [1]:
import json
from pathlib import Path
import pandas as pd

RESULTS = Path("../perf/results")

def load(name):
    return json.loads((RESULTS / name).read_text(encoding="utf-8"))

## Baseline

Mesures sur le même profil UI, en local et en préprod.

In [2]:
local = load("baseline_20260416_101611.json")
preprod = load("preprod_20260416_105012.json")

rows = []
for scope in ["inference", "service", "http"]:
    rows.append({
        "portée": scope,
        "local (médiane ms)": local["metrics"][scope]["stats"]["median_ms"],
        "préprod (médiane ms)": preprod["metrics"][scope]["stats"]["median_ms"],
    })
pd.DataFrame(rows)

,portée,local (médiane ms),préprod (médiane ms)
0,inference,0.514,0.497
1,service,0.737,0.720
2,http,121.810,372.247


Lecture :
- le moteur (`inference`, `service`) coûte moins d'une milliseconde ;
- le temps perçu utilisateur (`http`) est bien plus grand, et encore plus en préprod.

## Monitoring existant

Les prédictions sont loggées dans `prediction_logs` (PostgreSQL) avec leur `duration_ms`.
Un monitoring Streamlit en lit les indicateurs principaux.
Une démonstration de drift avec Evidently est dans `notebooks/01_monitoring_drift.ipynb`.

Ce notebook s'appuie sur ces données mais n'en refait pas l'analyse.

## Identification des goulots

cProfile sur 1000 appels de `score_client()` en local (voir `perf/bottlenecks_analysis.md`).

| Mesure | Valeur |
|---|---|
| Temps total cProfile | ~1.75 s |
| Par appel | ~1.75 ms |
| Part du calcul LightGBM pur | ~17 % du temps profilé |
| Overhead autour du modèle | sklearn (validation, transform pandas) + construction DataFrame |
| CPU | ~100 % d'un cœur (mono-thread) |
| RAM (peak RSS) | ~213 MB |

Pourquoi pas de GPU :
- LightGBM en inférence single-sample n'en tire pas parti ;
- Render ne fournit pas de GPU sur le plan utilisé ;
- le moteur n'est pas le goulot principal côté utilisateur.

## Optimisation moteur — ONNX Runtime

Conversion de la pipeline sklearn en ONNX, puis comparaison sur le même profil.

In [3]:
onnx = load("onnx_20260416_120544.json")

print(f"Ratio médiane sklearn / ONNX : ×{onnx['ratio_median_sklearn_over_onnx']}")
print(f"Écart de score (tolérance 1e-4) : {onnx['equivalence']['diff']:.2e}")

pd.DataFrame([
    {"moteur": "sklearn", "médiane (ms)": onnx["metrics"]["sklearn"]["stats"]["median_ms"]},
    {"moteur": "onnx",    "médiane (ms)": onnx["metrics"]["onnx"]["stats"]["median_ms"]},
])

Ratio médiane sklearn / ONNX : ×51.583
Écart de score (tolérance 1e-4) : 9.37e-09


,moteur,médiane (ms)
0,sklearn,0.619
1,onnx,0.012


Décision : **ONNX retenu comme optimisation moteur validée**.
Scores équivalents, inférence ~50× plus rapide — mais le gain absolu reste faible (~0.6 ms), donc seul il ne transforme pas le temps perçu utilisateur.

## Optimisation applicative — PostgreSQL asynchrone

Écriture des logs déplacée dans un `ThreadPoolExecutor`, activable par `ASYNC_DB_LOGGING=1`.
Intégrité vérifiée via filtre `environment + timestamp` côté DB.

In [4]:
pg_sync  = load("postgres_sync_20260416_124914.json")
pg_async = load("postgres_async_20260416_125817.json")
preprod_sync  = load("preprod_20260416_105012.json")
preprod_async = load("preprod_20260416_144314.json")

pd.DataFrame([
    {"environnement": "local",   "mode": "sync",  "HTTP médiane (ms)": pg_sync["metrics"]["http"]["stats"]["median_ms"],
     "logs": f"{pg_sync['log_integrity']['inserted']}/{pg_sync['log_integrity']['expected']}"},
    {"environnement": "local",   "mode": "async", "HTTP médiane (ms)": pg_async["metrics"]["http"]["stats"]["median_ms"],
     "logs": f"{pg_async['log_integrity']['inserted']}/{pg_async['log_integrity']['expected']}"},
    {"environnement": "préprod", "mode": "sync",  "HTTP médiane (ms)": preprod_sync["metrics"]["http"]["stats"]["median_ms"],
     "logs": "—"},
    {"environnement": "préprod", "mode": "async", "HTTP médiane (ms)": preprod_async["metrics"]["http"]["stats"]["median_ms"],
     "logs": "—"},
])

,environnement,mode,HTTP médiane (ms),logs
0,local,sync,1038.756,100/100
1,local,async,120.843,100/100
2,préprod,sync,372.247,—
3,préprod,async,380.158,—


Décision : **PostgreSQL async retenu comme optimisation applicative**.
- Gain très net en local (~918 ms gagnées en médiane).
- Drain complet des écritures en ~19 s pour 100 requêtes (aucune perte).
- Sur cette préprod Render, le gain n'est pas observé clairement (environnement partagé, variance d'infra).

## Synthèse finale

Chaque mesure vient d'un protocole précis. Les deux tableaux ci-dessous regroupent les runs réels **sans les mélanger dans une même ligne**, avec leur source.

**Moteur — mesures in-process local**

| Mesure           | médiane (ms) | Source |
|------------------|-------------:|--------|
| sklearn (baseline `inference`) |        0.514 | `baseline_20260416_101611.json` |
| sklearn (run ONNX)             |        0.619 | `onnx_20260416_120544.json` |
| onnx                           |        0.012 | `onnx_20260416_120544.json` |

Les deux lignes sklearn portent sur la même fonction (`model.predict_proba`) mesurée dans deux runs différents — la variance est normale.

**Applicatif — mesures HTTP**

| Environnement | Mode  | HTTP médiane (ms) | Source |
|---------------|-------|------------------:|--------|
| local         | sync  |          1038.756 | `postgres_sync_20260416_124914.json` |
| local         | async |           120.843 | `postgres_async_20260416_125817.json` |
| préprod       | sync  |           372.247 | `preprod_20260416_105012.json` |
| préprod       | async |           380.158 | `preprod_20260416_144314.json` |

Lecture :
- **ONNX** améliore le moteur (×50 environ, gain absolu ~0.6 ms), scores équivalents.
- **PostgreSQL async** améliore le chemin applicatif en local ; sur cette préprod Render, le gain n'est pas confirmé.
- Les deux optimisations agissent à des niveaux différents et sont complémentaires.
- Aucune ligne ne combine ONNX + PostgreSQL async : ONNX n'est pas intégré à l'application, la combinaison n'a pas été mesurée de bout en bout.

## Conclusion

- **Baseline** : le moteur est très rapide, le temps perçu vient surtout de la couche autour.
- **cProfile** : confirme que le modèle n'est pas le goulot principal.
- **ONNX Runtime** : retenu comme optimisation moteur validée.
- **PostgreSQL async** : retenu comme optimisation applicative, nette en local, non confirmée sur cette préprod Render.
- **Décision finale** : deux optimisations conservées, présentées comme complémentaires, avec lecture prudente de la préprod.